# Tree-Bound LLM: Native LLM-Tree Integration

**Core idea:** The tree IS the LLM's mathematical memory.

- Cross-attention from LLM hidden states to signature embeddings
- Welford variance controls attention temperature
- Tree grows dynamically during cold start
- Per-step LLM encoding for improved routing

**Architecture:**
```
Problem → TinyLlama (frozen) → Per-Step Encoding → Cross-Attention to Tree → Route → Execute
                                      ↑
                            Signature embeddings (learnable)
                            Welford stats (updated at runtime)
```

---

# Setup (Run in Browser First)

In [ ]:
# Use older transformers to avoid attention bugs
!pip install transformers==4.36.0 accelerate sentencepiece --quiet

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer
import json
import math
from typing import Optional, Dict, List, Tuple
from dataclasses import dataclass

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Upload GSM8K data
from google.colab import files
import os

os.makedirs('data', exist_ok=True)
print("Upload train.jsonl and test.jsonl from data/gsm8k_gts/")
uploaded = files.upload()
for name in uploaded:
    os.rename(name, f'data/{name}')
    print(f"Saved: data/{name}")

# 1. Core Components

## 1.1 Signature Tree (Storage)

## 1. Signature Storage (The Tree)

## 1.2 Tree-Bound LLM Model

Key methods:
- `encode_steps_batched()`: Per-step LLM encoding  
- `route_steps()`: Route each step to signatures
- `forward()`: Legacy single-embedding interface

## 2. Tree-Bound LLM Model

In [ ]:
class TreeCrossAttention(nn.Module):
    """Cross-attention layer from LLM hidden states to tree signatures."""
    
    def __init__(self, hidden_size: int, signature_dim: int, num_heads: int = 4):
        super().__init__()
        
        self.hidden_size = hidden_size
        self.signature_dim = signature_dim
        self.num_heads = num_heads
        self.head_dim = signature_dim // num_heads
        
        self.q_proj = nn.Linear(hidden_size, signature_dim)
        self.k_proj = nn.Linear(signature_dim, signature_dim)
        self.out_proj = nn.Linear(signature_dim, signature_dim)
        self.norm = nn.LayerNorm(signature_dim)
        
    def forward(
        self, 
        hidden_states: torch.Tensor,  # [batch, hidden_size]
        signature_embeddings: torch.Tensor,  # [num_sigs, sig_dim]
        welford_variances: torch.Tensor,  # [num_sigs]
        return_weights: bool = True
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Cross-attention to tree."""
        Q = self.q_proj(hidden_states)
        K = self.k_proj(signature_embeddings)
        
        scores = torch.matmul(Q, K.T) / math.sqrt(self.signature_dim)
        temperature = 1.0 + welford_variances.unsqueeze(0)
        adjusted_scores = scores / temperature
        attn_weights = F.softmax(adjusted_scores, dim=-1)
        
        return Q, attn_weights, scores


class TreeBoundLLM(nn.Module):
    """LLM with native tree binding via cross-attention."""
    
    def __init__(
        self, 
        base_model_name: str = "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
        signature_dim: int = 256,
        freeze_llm: bool = True,
        min_signatures_before_routing: int = 5,
    ):
        super().__init__()
        
        self.min_signatures_before_routing = min_signatures_before_routing
        
        print(f"Loading {base_model_name}...")
        self.llm = AutoModel.from_pretrained(base_model_name)
        self.tokenizer = AutoTokenizer.from_pretrained(base_model_name)
        
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        
        if freeze_llm:
            print("Freezing LLM parameters...")
            for param in self.llm.parameters():
                param.requires_grad = False
        
        hidden_size = self.llm.config.hidden_size
        self.signature_dim = signature_dim
        self.hidden_size = hidden_size
        
        self.tree_attention = TreeCrossAttention(
            hidden_size=hidden_size,
            signature_dim=signature_dim
        )
        
        self.base_threshold = nn.Parameter(torch.tensor(0.5))
        
        print(f"TreeBoundLLM initialized:")
        print(f"  LLM hidden size: {hidden_size}")
        print(f"  Signature dim: {signature_dim}")
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"  Trainable params: {trainable:,}")
    
    def encode_text(self, text: str, device) -> torch.Tensor:
        """Encode a single text to hidden state."""
        inputs = self.tokenizer(
            text,
            max_length=512,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        ).to(device)
        
        with torch.no_grad():
            outputs = self.llm(**inputs)
        
        return outputs.last_hidden_state[:, -1, :].float()
    
    def encode_steps_batched(
        self, 
        problem_text: str, 
        prefix_steps: List[str],
        num_map: Dict[str, str],
        device
    ) -> torch.Tensor:
        """
        Encode all steps from one problem in a single batched forward pass.
        
        Returns: [num_steps, hidden_size] tensor of step embeddings
        """
        if not prefix_steps:
            return None
        
        # Build step-specific texts
        step_texts = []
        for i, prefix in enumerate(prefix_steps):
            op, left, right = parse_prefix_step(prefix)
            
            # Resolve operand names to give more context
            left_val = num_map.get(left, left) if num_map else left
            right_val = num_map.get(right, right) if num_map else right
            
            # Format: problem + step context
            step_text = f"solve: {problem_text} [step {i+1}: compute {left_val} {op} {right_val}]"
            step_texts.append(step_text)
        
        # Batch tokenize
        inputs = self.tokenizer(
            step_texts,
            max_length=512,
            padding=True,
            truncation=True,
            return_tensors="pt"
        ).to(device)
        
        # Single batched forward pass
        outputs = self.llm(**inputs)
        
        # Get last token embedding for each step
        step_embeddings = outputs.last_hidden_state[:, -1, :].float()
        
        return step_embeddings  # [num_steps, hidden_size]
    
    def route_steps(
        self,
        step_embeddings: torch.Tensor,  # [num_steps, hidden_size]
        tree: 'SignatureTree',
        debug: bool = False
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Route each step embedding to signatures.
        
        Returns: (queries, route_probs, scores) each [num_steps, ...]
        """
        sig_embeddings = tree.get_active_embeddings().to(step_embeddings.device)
        variances = tree.get_variances().to(step_embeddings.device)
        
        queries, route_probs, scores = self.tree_attention(
            step_embeddings, sig_embeddings, variances
        )
        
        if debug:
            print(f"    route_steps: {step_embeddings.shape[0]} steps → {route_probs.shape}")
        
        return queries, route_probs, scores
    
    # Keep old interface for backwards compatibility
    def get_hidden_state(self, input_ids, attention_mask, debug=False) -> torch.Tensor:
        outputs = self.llm(input_ids=input_ids, attention_mask=attention_mask)
        return outputs.last_hidden_state[:, -1, :].float()
    
    def forward(self, input_ids, attention_mask, tree, debug=False):
        hidden = self.get_hidden_state(input_ids, attention_mask, debug)
        sig_embeddings = tree.get_active_embeddings().to(hidden.device)
        variances = tree.get_variances().to(hidden.device)
        query, route_probs, scores = self.tree_attention(hidden, sig_embeddings, variances)
        should_create = self.should_create_signature(tree, scores, debug)
        return query, route_probs, scores, should_create
    
    def should_create_signature(self, tree, scores, debug=False) -> bool:
        num_sigs = tree.num_active
        if num_sigs < self.min_signatures_before_routing:
            import random
            return random.random() < 0.7
        max_score = scores.max().item()
        threshold = self.base_threshold.detach().item()
        maturity = min(1.0, num_sigs / 50)
        return max_score < threshold + 0.3 * maturity


print("Model with per-step encoding ready.")

In [ ]:
# Initialize model and tree
model = TreeBoundLLM(
    base_model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    signature_dim=256,
    freeze_llm=True
).to(device)

tree = SignatureTree(embedding_dim=256).to(device)

print(f"\nTree has {tree.num_active} signatures")

## 1.3 DSL Execution

Prefix notation parsing and multi-step execution with result chaining.

## 3. DSL Execution

## 1.4 DSL Generation (Heuristic)

Simple keyword-based DSL generation. In production, this would use LLM.

## 4. Signature Generation (LLM creates DSL)

# 2. Training

## 2.1 Data Loading & Dataset

## 5. Training Loop with Tree Growth

## 2.2 Training Functions (Per-Step Encoding)

Key function: `train_step_per_step_encoding()` - each step gets its own embedding.

In [ ]:
def load_gsm8k_data(path: str) -> List[dict]:
    """Load GSM8K data."""
    with open(path) as f:
        return [json.loads(line) for line in f]


def extract_numbers_from_problem(problem: str) -> Dict[str, float]:
    """Extract numbers from problem text."""
    import re
    numbers = re.findall(r'\b\d+(?:\.\d+)?\b', problem)
    return {f"NUM_{i}": float(n) for i, n in enumerate(numbers)}


def prefix_op_to_dsl(op: str) -> str:
    """Map prefix operator to our DSL format."""
    op_map = {'+': 'a + b', '-': 'a - b', '*': 'a * b', '/': 'a / b'}
    return op_map.get(op, 'a + b')


class GSM8KDataset(Dataset):
    def __init__(self, data: List[dict], tokenizer, max_length=512):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        
        question = item.get("normalized_question", item.get("question", ""))
        text = f"solve: {question}"
        
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        
        # Get prefix_steps for multi-step execution
        prefix_steps = item.get("prefix_steps", [])
        
        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "answer": float(item.get("answer", 0)),
            "num_map": item.get("num_map", {}),
            "question": question,
            "prefix_steps": prefix_steps,  # List of atomic operations
            "num_steps": item.get("num_steps", 1),
        }


# Custom collate to handle variable-length prefix_steps
def collate_fn(batch):
    """Custom collate that handles prefix_steps lists."""
    return {
        "input_ids": torch.stack([b["input_ids"] for b in batch]),
        "attention_mask": torch.stack([b["attention_mask"] for b in batch]),
        "answer": [b["answer"] for b in batch],
        "num_map": [b["num_map"] for b in batch],
        "question": [b["question"] for b in batch],
        "prefix_steps": [b["prefix_steps"] for b in batch],
        "num_steps": [b["num_steps"] for b in batch],
    }

## 2.3 Run Training

Load data, create model/tree, run training loop.

In [ ]:
def train_epoch(
    model: TreeBoundLLM,
    tree: SignatureTree,
    dataloader: DataLoader,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    max_steps: int = None
) -> dict:
    """
    Train for one epoch.
    """
    total_loss = 0
    total_correct = 0
    total_created = 0
    n_steps = 0
    
    for i, batch in enumerate(dataloader):
        if max_steps and i >= max_steps:
            break
        
        # Debug first step only
        debug = (i == 0)
        
        loss, correct, created = train_step(
            model, tree, batch, optimizer, device, debug=debug
        )
        
        total_loss += loss
        total_correct += int(correct)
        total_created += int(created)
        n_steps += 1
        
        if (i + 1) % 50 == 0:
            acc = total_correct / n_steps
            print(f"  Step {i+1}: loss={total_loss/n_steps:.4f}, acc={acc:.1%}, sigs={tree.num_active}")
    
    return {
        "loss": total_loss / n_steps,
        "accuracy": total_correct / n_steps,
        "created": total_created,
        "num_signatures": tree.num_active
    }

## 6. Run Training (Cold Start)

In [ ]:
# Load ALL data (not just 1-step problems)
train_data = load_gsm8k_data("data/train.jsonl")
test_data = load_gsm8k_data("data/test.jsonl")

# Stats on step distribution
step_counts = {}
for ex in train_data:
    n = ex.get("num_steps", 1)
    step_counts[n] = step_counts.get(n, 0) + 1

print(f"Training data: {len(train_data)} problems")
print(f"Test data: {len(test_data)} problems")
print(f"\nStep distribution:")
for n in sorted(step_counts.keys()):
    print(f"  {n}-step: {step_counts[n]} ({100*step_counts[n]/len(train_data):.1f}%)")

In [ ]:
# Create fresh model and tree
model = TreeBoundLLM(
    base_model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    signature_dim=256,
    freeze_llm=True
).to(device)

tree = SignatureTree(embedding_dim=256, max_signatures=100).to(device)

# Dataset with ALL problems (multi-step)
train_dataset = GSM8KDataset(train_data, model.tokenizer)
train_loader = DataLoader(
    train_dataset, 
    batch_size=1, 
    shuffle=True,
    collate_fn=collate_fn  # Handle variable-length prefix_steps
)

# Optimizer
optimizer = torch.optim.AdamW(
    model.tree_attention.parameters(),
    lr=1e-4,
    weight_decay=0.01
)

print(f"Ready for MULTI-STEP training!")
print(f"Initial tree: {tree.num_active} signatures")
print(f"Problems: {len(train_data)} (multi-step)")

In [ ]:
# Per-step LLM encoding training
NUM_EPOCHS = 5
STEPS_PER_EPOCH = 300

history = []

print("="*60)
print("PER-STEP LLM ENCODING TRAINING")
print("Each step gets its own embedding: 'solve: {problem} [step N: compute X op Y]'")
print("This should improve routing accuracy from ~36% baseline")
print("="*60)

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
    print("-" * 40)
    
    metrics = train_epoch_per_step(
        model, tree, train_loader, optimizer, device,
        max_steps=STEPS_PER_EPOCH
    )
    
    history.append(metrics)
    
    print(f"\nEpoch {epoch+1} complete:")
    print(f"  Loss: {metrics['loss']:.4f}")
    print(f"  Answer accuracy: {metrics['answer_accuracy']:.1%}")
    print(f"  Routing accuracy: {metrics['routing_accuracy']:.1%}  <-- TARGET: >50%")
    print(f"  Avg steps/problem: {metrics['avg_steps']:.2f}")

print("\n" + "="*60)
print("PER-STEP ENCODING TRAINING COMPLETE")
print(f"Final routing accuracy: {history[-1]['routing_accuracy']:.1%}")
print(f"Previous baseline was ~36% with single embedding")
print("="*60)

# 3. Evaluation & Results

In [ ]:
# Inspect the tree using summary()
print("="*60)
print("SIGNATURE TREE SUMMARY (Deduplicated by DSL)")
print("="*60)
print(tree.summary())

print("\n" + "="*60)
print("DETAILED SIGNATURES")
print("="*60)
for idx in range(tree.num_active):
    sig = tree.signatures[idx]
    print(f"\n[{idx}] {sig.dsl}")
    print(f"    Description: {sig.description}")
    print(f"    Usage: n={sig.n}, success={sig.mean:.1%}, variance={sig.variance:.3f}")

## 7. Evaluation

# 4. Interactive Testing

Try custom problems to see routing in action.

In [ ]:
# Save model and tree
import pickle

torch.save({
    "tree_attention": model.tree_attention.state_dict(),
    "base_threshold": model.base_threshold,
}, "tree_bound_llm.pt")

with open("signature_tree.pkl", "wb") as f:
    pickle.dump({
        "signatures": tree.signatures,
        "embeddings": tree.embeddings.cpu(),
        "active_mask": tree.active_mask.cpu(),
        "num_active": tree.num_active,
    }, f)

print("Saved model and tree!")

# Download
files.download("tree_bound_llm.pt")
files.download("signature_tree.pkl")

## 8. Interactive Testing

In [ ]:
@torch.no_grad()
def solve_problem(model, tree, problem, device):
    """Solve a single problem."""
    model.eval()
    
    # Extract numbers
    numbers = extract_numbers_from_problem(problem)
    print(f"Numbers found: {numbers}")
    
    # Tokenize
    text = f"solve: {problem}"
    inputs = model.tokenizer(text, return_tensors="pt", padding=True).to(device)
    
    # Forward
    query, route_probs, scores, should_create = model(
        inputs["input_ids"],
        inputs["attention_mask"],
        tree
    )
    
    # Show routing
    print(f"\nRouting probabilities:")
    for idx in range(tree.num_active):
        prob = route_probs[0, idx].item()
        sig = tree.signatures[idx]
        print(f"  [{idx}] {sig.dsl:10s} : {prob:.1%}")
    
    # Route
    route_idx = route_probs.argmax(dim=-1).item()
    dsl = tree.get_dsl(route_idx)
    print(f"\nSelected: Signature {route_idx} ({dsl})")
    
    # Execute
    operands = {"a": list(numbers.values())[0] if numbers else 0,
                "b": list(numbers.values())[1] if len(numbers) > 1 else 0}
    result = execute_dsl(dsl, operands, {})
    print(f"Result: {result}")
    
    return result


# Try it!
solve_problem(model, tree, "John has 5 apples. Mary gives him 3 more. How many apples?", device)

In [ ]:
# Try more problems
problems = [
    "There are 12 cookies. 4 kids share them equally. How many does each get?",
    "A book costs 15 dollars. You have 8 dollars. How much more do you need?",
    "There are 6 rows with 7 chairs each. How many chairs total?",
]

for p in problems:
    print("\n" + "="*60)
    print(f"Problem: {p}")
    solve_problem(model, tree, p, device)